In [29]:
# Importing the required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [30]:
# Loading the various data files

duration = pd.read_csv('../data/raw/actual_duration.csv')
appointments = pd.read_csv('../data/raw/appointments_regional.csv')
regions = pd.read_excel('../data/raw/ICB_Region_Codes.xlsx')
categories = pd.read_excel('../data/raw/national_categories.xlsx')
tweets = pd.read_csv('../data/raw/tweets.csv')

In [31]:
# Checking the info of each df

duration.info()
appointments.info()
regions.info()
categories.info()
tweets.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 137793 entries, 0 to 137792
Data columns (total 8 columns):
 #   Column                     Non-Null Count   Dtype 
---  ------                     --------------   ----- 
 0   sub_icb_location_code      137793 non-null  object
 1   sub_icb_location_ons_code  137793 non-null  object
 2   sub_icb_location_name      137793 non-null  object
 3   icb_ons_code               137793 non-null  object
 4   region_ons_code            137793 non-null  object
 5   appointment_date           137793 non-null  object
 6   actual_duration            137793 non-null  object
 7   count_of_appointments      137793 non-null  int64 
dtypes: int64(1), object(7)
memory usage: 8.4+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 596821 entries, 0 to 596820
Data columns (total 7 columns):
 #   Column                             Non-Null Count   Dtype 
---  ------                             --------------   ----- 
 0   icb_ons_code                       59

In [32]:
# Changing the data types to ensure they correctly reflect the data
duration['appointment_date'] = pd.to_datetime(duration['appointment_date'])

appointments['appointment_month'] = pd.to_datetime(appointments['appointment_month'])



C:\Users\Charl\AppData\Local\Temp\ipykernel_18244\847304001.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  duration['appointment_date'] = pd.to_datetime(duration['appointment_date'])


In [33]:
duration.info()
appointments.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 137793 entries, 0 to 137792
Data columns (total 8 columns):
 #   Column                     Non-Null Count   Dtype         
---  ------                     --------------   -----         
 0   sub_icb_location_code      137793 non-null  object        
 1   sub_icb_location_ons_code  137793 non-null  object        
 2   sub_icb_location_name      137793 non-null  object        
 3   icb_ons_code               137793 non-null  object        
 4   region_ons_code            137793 non-null  object        
 5   appointment_date           137793 non-null  datetime64[ns]
 6   actual_duration            137793 non-null  object        
 7   count_of_appointments      137793 non-null  int64         
dtypes: datetime64[ns](1), int64(1), object(6)
memory usage: 8.4+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 596821 entries, 0 to 596820
Data columns (total 7 columns):
 #   Column                             Non-Null Count   Dtype         

In [34]:
# Checking for duplicated values in the data
duration.duplicated().value_counts()

False    137793
Name: count, dtype: int64

In [35]:
regions.duplicated().value_counts()

False    7
Name: count, dtype: int64

In [36]:
categories.duplicated().value_counts()

False    817394
Name: count, dtype: int64

In [37]:
appointments.duplicated().value_counts()

False    575217
True      21604
Name: count, dtype: int64

In [38]:
tweets.duplicated().value_counts()

False    1174
Name: count, dtype: int64

As the duplicates amount to 3.6% of the data, I've decided to remove them, as they won't materially impact the overall integrity of the data.

In [39]:
apps_clean = appointments.drop_duplicates().reset_index(drop=True)

In [40]:
# Checking the categorical values in duration data

# Actual duration
duration['actual_duration'].unique()

array(['31-60 Minutes', '21-30 Minutes', '6-10 Minutes',
       'Unknown / Data Quality', '16-20 Minutes', '11-15 Minutes',
       '1-5 Minutes'], dtype=object)

In [41]:
# Creating an order list so that when I come to visualise the duration, there order is pre determined
duration_order = ['1-5 Minutes', '6-10 Minutes', '11-15 Minutes', '16-20 Minutes', '21-30 Minutes', '31-60 Minutes', 'Unknown / Data Quality']

In [42]:
# Checking categorical values in appointments data

# Appointmnet status
apps_clean['appointment_status'].unique()

array(['Attended', 'DNA', 'Unknown'], dtype=object)

In [43]:
# Appointment mode
apps_clean['appointment_mode'].unique()

array(['Face-to-Face', 'Home Visit', 'Telephone', 'Unknown',
       'Video/Online'], dtype=object)

In [44]:
# HCP Type
apps_clean['hcp_type'].unique()

array(['GP', 'Other Practice staff', 'Unknown'], dtype=object)

In [45]:
# Time between booking and attending an appointment
apps_clean['time_between_book_and_appointment'].unique()

array(['1 Day', '15  to 21 Days', '2 to 7 Days', '22  to 28 Days',
       '8  to 14 Days', 'More than 28 Days', 'Same Day',
       'Unknown / Data Quality'], dtype=object)

In [46]:
# Standardising the formatting of the time between booking and attending (double spaces)
apps_clean['time_between_book_and_appointment'] = apps_clean['time_between_book_and_appointment'].str.replace(r'\s', ' ', regex=True)

In [47]:
# Checking changes
apps_clean['time_between_book_and_appointment'].unique()

array(['1 Day', '15  to 21 Days', '2 to 7 Days', '22  to 28 Days',
       '8  to 14 Days', 'More than 28 Days', 'Same Day',
       'Unknown / Data Quality'], dtype=object)

In [48]:
# Like the app duration, creating an order for the time between booking and attending ready for visualisation
booking_order = ['Same Day', '1 Day', '2 to 7 Days', '8 to 14 Days', '15 to 21 Days', '22 to 28 Days', 'More than 28 Days', 'Unknown / Data Quality']

In [49]:
# Checking categorical values in categories data

# Service setting
categories['service_setting'].unique()

array(['Primary Care Network', 'Other', 'General Practice', 'Unmapped',
       'Extended Access Provision'], dtype=object)

In [50]:
# Context type
categories['context_type'].unique()

array(['Care Related Encounter', 'Unmapped', 'Inconsistent Mapping'],
      dtype=object)

In [51]:
categories['national_category'].unique()

array(['Patient contact during Care Home Round', 'Planned Clinics',
       'Home Visit', 'General Consultation Acute',
       'Structured Medication Review', 'Care Home Visit', 'Unmapped',
       'Clinical Triage', 'Planned Clinical Procedure',
       'Inconsistent Mapping',
       'Care Home Needs Assessment & Personalised Care and Support Planning',
       'General Consultation Routine',
       'Service provided by organisation external to the practice',
       'Unplanned Clinical Activity', 'Social Prescribing Service',
       'Non-contractual chargeable work',
       'Group Consultation and Group Education', 'Walk-in'], dtype=object)

In [52]:
# Final null check across data post any cleaning
apps_clean.isna().sum()

icb_ons_code                         0
appointment_month                    0
appointment_status                   0
hcp_type                             0
appointment_mode                     0
time_between_book_and_appointment    0
count_of_appointments                0
dtype: int64

In [53]:
duration.isna().sum()

sub_icb_location_code        0
sub_icb_location_ons_code    0
sub_icb_location_name        0
icb_ons_code                 0
region_ons_code              0
appointment_date             0
actual_duration              0
count_of_appointments        0
dtype: int64

In [54]:
categories.isna().sum()

appointment_date         0
icb_ons_code             0
sub_icb_location_name    0
service_setting          0
context_type             0
national_category        0
count_of_appointments    0
appointment_month        0
dtype: int64

In [55]:
regions.isna().sum()

NHSER22CD    0
NHSER22NM    0
dtype: int64

In [56]:
tweets.isna().sum()

tweet_id                     0
tweet_full_text              0
tweet_entities               0
tweet_entities_hashtags    167
tweet_metadata               0
tweet_retweet_count          0
tweet_favorite_count         0
tweet_favorited              0
tweet_retweeted              0
tweet_lang                   0
dtype: int64

In [61]:
# Loading the cleaned and formatted data to a new csv file in the data/clean folder 
duration.to_csv('../data/clean/dur_clean.csv')
apps_clean.to_excel('../data/clean/apps_clean.xlsx')
regions.to_csv('../data/clean/reg_clean.csv')
categories.to_excel('../data/clean/cat_clean.xlsx')
tweets.to_csv('../data/clean/tweet_clean.csv')